In [1]:

from datetime import datetime, timedelta
import math
import pandas as pd
import numpy as np

In [2]:


df_procesar = pd.read_csv(
    "../data/raw/ventas_completo.csv",
    usecols=['identification_doct', 'product', 'date_sale'])

# Renombrar y filtrar IDs de cliente
df_procesar['client'] = df_procesar['identification_doct'].astype(str).str.strip()
mask_digits = df_procesar["client"].str.isdigit().fillna(False)
mask_zero = ~df_procesar["client"].str.startswith("0", na=False)
df_procesar = df_procesar[mask_digits & mask_zero].copy()
print(f"Filas tras filtrar id_client no numéricos/cero inicial: {len(df_procesar)}")

# Limpiar fechas, productos domicilio
df_procesar['product'] = df_procesar['product'].astype(str).str.strip()
df_procesar['date_sale'] = pd.to_datetime(df_procesar['date_sale'])
df_procesar = df_procesar.dropna(subset=['date_sale'])
df_procesar['date_sale'] = df_procesar['date_sale'].dt.normalize()


productos = pd.read_csv(
    f"../data/raw/productos.csv",
    dtype={"codigo_unico": str}
).rename(columns={"codigo_unico": "product"})
productos = productos.loc[:, ["product", "description", "brand", "category"]]
productos = productos.drop_duplicates("product")
productos['product'] = productos['product'].str.strip()
df_procesar = pd.merge(df_procesar, productos, on="product")
print(f"Datos cargados Filas: {len(df_procesar)}")


predictions = pd.read_csv("../predictions/predictions.csv", converters={"client": str})


/var/folders/1g/77kw2x4j5678s_87_sqc1fpc0000gp/T/ipykernel_75374/991939786.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_procesar = pd.read_csv(


Filas tras filtrar id_client no numéricos/cero inicial: 18407577
Datos cargados Filas: 18385402


In [27]:
LAST_MONTH = 1
TOP_PCT = 0.75
EXCLUIR = [
    'BOLSA PLASTICA',
    'BOLSA RECUPERADA',
    'MENU EMPLEADOS EURO',
]

In [28]:

print(f"Filtrando clientes con compras en los últimos {LAST_MONTH} meses...")
last_months_ago = datetime.now().date() - timedelta(days=LAST_MONTH*30)
recent_purchases = df_procesar[df_procesar["date_sale"].dt.date >= last_months_ago]
recent_purchases = recent_purchases[~recent_purchases["description"].isin(EXCLUIR)]
print(f"Compras en los últimos {LAST_MONTH} meses: {len(recent_purchases)}")
hist_recent_client = recent_purchases[recent_purchases["client"].isin(predictions["client"])]
print(f"Clientes predichos con compras en los últimos {LAST_MONTH} meses: {hist_recent_client["client"].nunique()}")
hist_recent_client["date_sale"].min(), hist_recent_client["date_sale"].max()

Filtrando clientes con compras en los últimos 1 meses...
Compras en los últimos 1 meses: 2561192
Clientes predichos con compras en los últimos 1 meses: 459


(Timestamp('2025-04-17 00:00:00'), Timestamp('2025-05-15 00:00:00'))

In [29]:
# Conteo de compras por cliente-producto
cant_compras_client = (
    hist_recent_client
    .groupby(["client", "description"])
    .size()                               # cuenta filas
    .rename("cant_compras")
    .reset_index()
)
#Ordenar de mayor a menor para cada cliente
cant_compras_client = (
    cant_compras_client
    .sort_values(["client", "cant_compras"], ascending=[True, False])
)

def top_pct(df):
    k = math.ceil(len(df) * TOP_PCT)
    return df.head(k)

top_products = (
    cant_compras_client
      .groupby("client", group_keys=False)
      .apply(top_pct)
      .reset_index(drop=True)
)

In [30]:
top_summary = (
    top_products
    .groupby('client')
    .agg(
        productos_comunes=('description', lambda x: ', '.join(x)),
        cantidad_productos=('description', 'count')
    )
    .reset_index()
)
top_summary

,client,productos_comunes,cantidad_productos
0,1000192661,"BUNUELO EURO 55 gr, PONY MALTA PET x 330ML, T...",27
1,1000410009,"BRETAÑA FRIOPACK SUPER *300ML, BEBIDA ENERGIZA...",47
2,1000440285,"GASEOSA COCACOLA *1.5L, COCACOLA x 600ML, GA...",31
3,1000445397,"ACOMPAÑAMIENTO COMIDA PPAL, BEBIDA ENERGIZAN B...",18
4,1000534537,"AGUA LIMA/LIMON HOJAS MENTA ZEN*540ML, MR TEA ...",72
...,...,...,...
454,98544040,"LECHE COLANTA SEMIDESCREM 1000ml, CIGARRILLO C...",25
455,98552891,"AGUA POTABLE TRATADA EUROMAX *1000ML, TAMPICO ...",61
456,98558712,"AREPA TELA BLANCA CASERA SARY X10 UNDS, QUESIT...",20
457,98630133,"AGUA BRISA LIMA LIMON PET*1500ML, AGUA H2O LIM...",19


In [31]:
predictions_with_precom = pd.merge(predictions, top_summary, on="client", how="left")

In [32]:
predictions_with_precom

,date,client,prob,name,email,phone,telephone,productos_comunes,cantidad_productos
0,2025-05-15,70569726,0.7463,OTALVARO GONZALEZ LUIS FERNANDO,luisfernandootalvarogonzalez@hotmail.com,3.108321e+09,0.000000e+00,"CIGARRILLO CHESTERFIRLD BLUE BOX x 20UND, CIGA...",81
1,2025-05-15,800180330,0.7463,COMPAÑIA DE ALIMENTOS COLOMBIANOS CALCO S.A,proveedores.fe.medellin@crepesywaffles.com,3.102119e+09,0.000000e+00,"PLATANO, PULPA EUROMAX CONGELADA DE MANGO X 5...",158
2,2025-05-15,1235245925,0.7350,GISEL FERNANDEZ,gisellecfr@gmail.com,3.013476e+09,0.000000e+00,"PECHUGA BUCANERO CAMPESINA REFRIGERADA, ARROZ ...",66
3,2025-05-15,7797719,0.7350,BUTSU WAKE,wake.butsu@gmail.com,3.136017e+09,NaN,"AGUACATE HASS, FILETE DE PECHUGA CAMPESINA, Z...",47
4,2025-05-15,901569974,0.7350,PATAKUS FOOD SAS,facturaspatakusfoodsas@gmail.com,3.015186e+09,0.000000e+00,"COSTILLA DE RES.QB, TRANSPORTE DOMICILIO, HUES...",115
...,...,...,...,...,...,...,...,...,...
454,2025-05-15,1036956908,0.5017,CONTRERAS HOYOS LENIS,lenicontrerashoyos@hotmail.com,3.132863e+09,3.132863e+09,"MANZANA ROYAL GALA, AGUACATE INJERTO, MANI KRA...",26
455,2025-05-15,1140887977,0.5017,GONZALEZ ADISON,adisongonzalezba@gmail.com,3.246062e+09,0.000000e+00,"BEBIDA ENERGIZANTE AMPER REG X473ML, GASEOSA C...",42
456,2025-05-15,34967510,0.5008,NEGRETE YENI,facturacionelectronicapos@eurosupermercados.com,NaN,0.000000e+00,"TOMATE CHONTO, MUSLOS FRESCOS, CEBOLLA BLANCA...",45
457,2025-05-15,43688000,0.5008,PAULA RESTREPO,facturacionelectronicapos@eurosupermercados.com,NaN,2.786382e+06,"BANANO CRIOLLO, ENERGIZANTE SPARTAN XTREME*240...",38


In [41]:
hist_recent_client[hist_recent_client["client"] == "800180330"]["description"].nunique()

210